# 03 - The Librarian (Advanced RAG)

## Setup & Dependencies

In [ ]:
# Install Weaviate & Dependencies
!pip install -q -U "weaviate-client<4.0.0" pypdf
!pip install -q -U langchain langchain-community langchain-text-splitters langchain-google-genai sentence-transformers python-dotenv

import os
repo_name = "zuucrew-ai-aee-mini-project-1"
repo_url = f"https://github.com/Sulamaxx/{repo_name}.git"

if 'google.colab' in str(get_ipython()):
    if not os.path.exists(f"/content/{repo_name}"):
        print(f"Cloning {repo_name}...")
        !git clone {repo_url}
    
    os.chdir(f"/content/{repo_name}")
    print(f"Switched to repo directory: {os.getcwd()}")
    
    try:
        from google.colab import userdata
        for key in ["GEMINI_API_KEY", "GOOGLE_API_KEY"]:
            # val = userdata.get(key)
            if val:
                os.environ[key] = val
                print(f"{key} loaded from Colab Secrets.")
                break
    except Exception:
        pass


In [ ]:
# Initialize LLM & Embeddings
from dotenv import load_dotenv
from src.services.llm_services import load_config, get_llm, get_text_embeddings

load_dotenv()
config = load_config("src/config/config.yaml")

llm = get_llm(config)
embeddings = get_text_embeddings(config)

print(f"LLM ({config['llm_model']}) and Embeddings initialized.")

## Ingestion & Chunking

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_path = "data/pdfs/2024AnnualReport.pdf"

if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"{pdf_path} not found!")

loader = PyPDFLoader(pdf_path)
pages = loader.load()

# Chunking: 1500 characters with 150 char overlap
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150,
    add_start_index=True
)
chunks = text_splitter.split_documents(pages)

print(f"PDF loaded and split into {len(chunks)} chunks.")


## Initialize Weaviate

In [ ]:
import weaviate
from weaviate.embedded import EmbeddedOptions
from tqdm.auto import tqdm

# Initialize Weaviate Client
client = weaviate.Client(
    embedded_options=EmbeddedOptions(),
    additional_headers={
        "X-Goog-Api-Key": os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    }
)
client.schema.delete_all()
print("Weaviate client initialized (Embedded Mode).")

## Define Schema & Index Chunks

In [ ]:
# Create Schema
schema = {
    "class": "LibrarianChunk",
    "vectorizer": "none", 
    "properties": [
        {"name": "text", "dataType": ["text"]},
        {"name": "page_number", "dataType": ["int"]}
    ]
}
client.schema.create_class(schema)

# Indexing
print("Indexing chunks into Weaviate...")
for i, chunk in enumerate(tqdm(chunks)):
    vector = embeddings.embed_query(chunk.page_content)
    client.data_object.create(
        data_object={
            "text": chunk.page_content,
            "page_number": chunk.metadata.get("page", 0)
        },
        class_name="LibrarianChunk",
        vector=vector
    )

print(f"Indexed {len(chunks)} chunks.")

## Hybrid Search & Reranking

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def hybrid_search(query, limit=10):
    query_vec = embeddings.embed_query(query)
    results = (client.query
               .get("LibrarianChunk", ["text", "page_number"])
               .with_hybrid(query=query, vector=query_vec, alpha=0.5)
               .with_limit(limit)
               .do())
    return results["data"]["Get"]["LibrarianChunk"]

def query_librarian(question):
    """RAG pipeline: Hybrid Search -> Rerank -> Final Generation."""
    # Retrieval (Hybrid)
    initial_results = hybrid_search(question, limit=10)
    if not initial_results: return "No info found."
    
    # Reranking
    passages = [res["text"] for res in initial_results]
    scores = reranker.predict([[question, p] for p in passages])
    ranked = sorted(zip(initial_results, scores), key=lambda x: x[1], reverse=True)
    
    best_context = ranked[0][0]["text"]
    page = ranked[0][0]["page_number"]
    
    # Generation
    prompt = f"""You are a precise librarian auditing Uber. Answer based ONLY on context.
    
    Context (Source Page {page}):
    {best_context}
    
    Question: {question}
    
    Answer:"""
    
    response = llm.invoke(prompt)
    return response.content if hasattr(response, 'content') else str(response)

test_q = "What were Uber's total revenues in 2024?"
print(f"Question: {test_q}")
print(f"\nLibrarian Answer:\n{query_librarian(test_q)}")